# catchPhish — ML Model Training

Trains a **Random Forest** classifier to detect phishing URLs using 16 URL-based features.

**Run all cells top-to-bottom. The final cell downloads `phishing_rf_model.pkl` to your computer.**

---
**Outputs you will see:**
- Dataset statistics and class distribution
- Confusion matrix heatmap
- Classification report (precision, recall, F1)
- Comparison of all 3 models (RF, XGBoost, Decision Tree)
- Feature importance bar chart
- Saved model files + auto-download

In [ ]:
# Cell 1 — Install dependencies
!pip install -q scikit-learn xgboost pandas numpy matplotlib seaborn joblib

In [ ]:
# Cell 2 — Upload or load dataset
# Option A: Upload from local machine
# from google.colab import files
# uploaded = files.upload()  # upload phishing_dataset.csv

# Option B: Download dataset directly (UCI ML Repository mirror)
import urllib.request, os
os.makedirs('data', exist_ok=True)

DATASET_URL = 'https://raw.githubusercontent.com/zNishanth/CommunityPrj/main/data/phishing_dataset.csv'
urllib.request.urlretrieve(DATASET_URL, 'data/phishing_dataset.csv')
print('Dataset downloaded successfully.')

In [ ]:
# Cell 3 — Load & inspect dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/phishing_dataset.csv')

print('=' * 50)
print(f'Dataset shape: {df.shape}')
print('=' * 50)
print(df.head(10).to_string())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nClass distribution:')
print(df['Label'].value_counts())
print(f"\nPhishing URLs  : {(df['Label'] == 1).sum()}")
print(f"Legitimate URLs: {(df['Label'] == 0).sum()}")

# Class distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Label'].value_counts().plot.pie(
    ax=axes[0], labels=['Legitimate', 'Phishing'],
    autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=90
)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('')

df['Label'].value_counts().plot.bar(ax=axes[1], color=['#4CAF50', '#F44336'])
axes[1].set_title('URL Count by Class')
axes[1].set_xticklabels(['Legitimate (0)', 'Phishing (1)'], rotation=0)
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print('Chart saved as class_distribution.png')

In [ ]:
# Cell 4 — Preprocessing & train/test split
from sklearn.model_selection import train_test_split

if 'Domain' in df.columns:
    df = df.drop(columns=['Domain'])

X = df.drop(columns=['Label'])
y = df['Label']

print('Feature columns:', list(X.columns))
print(f'Total features : {X.shape[1]}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain samples  : {len(X_train)}')
print(f'Test samples   : {len(X_test)}')

# Feature correlation heatmap
plt.figure(figsize=(14, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()
print('Correlation heatmap saved.')

In [ ]:
# Cell 5 — Train Random Forest (primary model)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)

print('=' * 50)
print(f'Random Forest Accuracy: {rf_acc * 100:.2f}%')
print('=' * 50)

# Confusion matrix
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'])
plt.title(f'Random Forest — Confusion Matrix\nAccuracy: {rf_acc*100:.2f}%')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved as rf_confusion_matrix.png')

print('\nClassification Report:')
print(classification_report(y_test, rf_pred,
                             target_names=['Legitimate', 'Phishing']))

In [ ]:
# Cell 6 — Compare all 3 models: RF vs XGBoost vs Decision Tree
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_acc  = accuracy_score(y_test, dt_pred)

xgb_model = XGBClassifier(
    n_estimators=100, learning_rate=0.1,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_acc  = accuracy_score(y_test, xgb_pred)

print('=' * 50)
print('MODEL COMPARISON')
print('=' * 50)
models     = ['Decision Tree', 'Random Forest', 'XGBoost']
accuracies = [dt_acc, rf_acc, xgb_acc]
for m, a in zip(models, accuracies):
    print(f'{m:20s}: {a*100:.2f}%')

print('\nDecision Tree Report:')
print(classification_report(y_test, dt_pred, target_names=['Legitimate','Phishing']))
print('XGBoost Report:')
print(classification_report(y_test, xgb_pred, target_names=['Legitimate','Phishing']))

# Bar chart comparison
plt.figure(figsize=(8, 5))
bars = plt.bar(models, [a * 100 for a in accuracies],
               color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='white', width=0.5)
plt.ylim(90, 100)
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             f'{acc*100:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
print('Model comparison chart saved as model_comparison.png')

In [ ]:
# Cell 7 — Feature importance chart (Random Forest)
feature_names = list(X.columns)
importances   = rf_model.feature_importances_
indices       = np.argsort(importances)[::-1]

print('Feature Importances (Random Forest):')
for i in indices:
    print(f'  {feature_names[i]:20s}: {importances[i]:.4f}')

plt.figure(figsize=(10, 6))
palette = sns.color_palette('viridis', len(feature_names))
bars = plt.barh(
    [feature_names[i] for i in indices[::-1]],
    importances[indices[::-1]],
    color=palette
)
plt.xlabel('Importance Score')
plt.title('Feature Importance — Random Forest Classifier')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print('Feature importance chart saved as feature_importance.png')

In [ ]:
# Cell 8 — Save trained model
import pickle, joblib, os
os.makedirs('models', exist_ok=True)

with open('models/phishing_rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print('Saved: models/phishing_rf_model.pkl')

joblib.dump(rf_model, 'models/phishing_rf_model.joblib')
print('Saved: models/phishing_rf_model.joblib')

print(f'\nFinal model: Random Forest')
print(f'Accuracy   : {rf_acc * 100:.2f}%')
print(f'Estimators : {rf_model.n_estimators}')
print(f'Features   : {X.shape[1]}')

In [ ]:
# Cell 9 — Download model file to your computer
from google.colab import files
files.download('models/phishing_rf_model.pkl')
print('Download started — place the file at:')
print('  backend/ml/models/phishing_rf_model.pkl')